## Step 0: Import libraries

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None, "display.width", 200)

## Step 1: Create a synthetic *dirty* dataset
Missing values and 12 duplicate rows are injected deliberately.

In [2]:
rng = np.random.default_rng(42)
n = 180
df = pd.DataFrame({
    "Order ID": np.arange(1001, 1001 + n),
    "Customer Name": [f"Customer_{i:03d}" for i in rng.integers(1, 60, n)],
    "City": rng.choice(["Surat", "Ahmedabad", "Mumbai", "Pune", "Vadodara", "Rajkot"], n),
    "Product Category": rng.choice(["Electronics", "Clothing", "Groceries", "Furniture", "Books"], n),
    "Quantity": rng.integers(1, 10, n).astype(float),
    "Unit Price": np.round(rng.uniform(100, 5000, n), 2),
    "Discount %": rng.choice([0, 5, 10, 15, 20], n).astype(float),
    "Order Date": pd.date_range("2025-01-01", periods=n, freq="D").strftime("%Y-%m-%d"),
})
for col, k in [("Quantity", 12), ("Unit Price", 10), ("Discount %", 9), ("City", 8)]:
    df.loc[rng.choice(n, k, replace=False), col] = np.nan      # inject missing values
df = pd.concat([df, df.sample(12, random_state=1)])             # inject 12 duplicate rows
df = df.sample(frac=1, random_state=2).reset_index(drop=True)

print("Original shape:", df.shape)
df.head()

Original shape: (192, 8)


,Order ID,Customer Name,City,Product Category,Quantity,Unit Price,Discount %,Order Date
0,1024,Customer_055,Vadodara,Clothing,2.0,2448.91,0.0,2025-01-24
1,1124,Customer_028,Rajkot,Furniture,2.0,1653.50,15.0,2025-05-04
2,1036,Customer_004,Ahmedabad,Books,4.0,2326.16,10.0,2025-02-05
3,1011,Customer_032,Vadodara,Clothing,8.0,3836.41,10.0,2025-01-11
4,1049,Customer_040,NaN,Books,8.0,621.46,10.0,2025-02-18


## Step 2: Identify missing values – `isna().sum()`

In [3]:
df.isna().sum()

Order ID             0
Customer Name        0
City                 9
Product Category     0
Quantity            13
Unit Price          11
Discount %          10
Order Date           0
dtype: int64

## Step 3: Option A – drop rows with missing values – `dropna()`

In [4]:
df_dropped = df.dropna()
print(f"dropna(): {df.shape[0]} -> {df_dropped.shape[0]} rows ({df.shape[0] - df_dropped.shape[0]} rows lost)")

dropna(): 192 -> 157 rows (35 rows lost)


## Step 4: Option B – impute missing values – `fillna()`
Numerical columns use mean/median; the categorical column uses the mode.

In [5]:
df_imputed = df.copy()
df_imputed["Quantity"]   = df_imputed["Quantity"].fillna(df_imputed["Quantity"].median())
df_imputed["Unit Price"] = df_imputed["Unit Price"].fillna(df_imputed["Unit Price"].mean())
df_imputed["Discount %"] = df_imputed["Discount %"].fillna(df_imputed["Discount %"].median())
df_imputed["City"]       = df_imputed["City"].fillna(df_imputed["City"].mode()[0])

print("Missing values after imputation:")
df_imputed.isna().sum()

Missing values after imputation:


Order ID            0
Customer Name       0
City                0
Product Category    0
Quantity            0
Unit Price          0
Discount %          0
Order Date          0
dtype: int64

## Step 5: Remove duplicate records – `drop_duplicates()`

In [6]:
print("Duplicate rows found:", df_imputed.duplicated().sum())
df_clean = df_imputed.drop_duplicates().reset_index(drop=True)
print("Shape after drop_duplicates():", df_clean.shape)

Duplicate rows found: 12
Shape after drop_duplicates(): (180, 8)


## Step 6: Rename columns – `rename()`

In [7]:
df_clean = df_clean.rename(columns={
    "Order ID": "order_id",
    "Customer Name": "customer_name",
    "City": "city",
    "Product Category": "product_category",
    "Quantity": "quantity",
    "Unit Price": "unit_price",
    "Discount %": "discount_pct",
    "Order Date": "order_date",
})
print(df_clean.columns.tolist())
df_clean.head()

['order_id', 'customer_name', 'city', 'product_category', 'quantity', 'unit_price', 'discount_pct', 'order_date']


,order_id,customer_name,city,product_category,quantity,unit_price,discount_pct,order_date
0,1024,Customer_055,Vadodara,Clothing,2.0,2448.91,0.0,2025-01-24
1,1124,Customer_028,Rajkot,Furniture,2.0,1653.50,15.0,2025-05-04
2,1036,Customer_004,Ahmedabad,Books,4.0,2326.16,10.0,2025-02-05
3,1011,Customer_032,Vadodara,Clothing,8.0,3836.41,10.0,2025-01-11
4,1049,Customer_040,Ahmedabad,Books,8.0,621.46,10.0,2025-02-18
